# ParPTE · Speaking model training (pronunciation + fluency)

Trains the models the ParPTE server uses to refine **Pronunciation** and **Oral Fluency** scores.

**Data:** [speechocean762](https://huggingface.co/datasets/mispeech/speechocean762), 5,000 English utterances from non-native speakers, each rated by experts for *accuracy* (pronunciation), *fluency*, *prosody* and *total* on a 0–10 scale. It is released under CC BY 4.0 on OpenSLR (SLR101), which allows commercial use with attribution. Confirm the licence on the dataset page before you launch.

* **Part A (≈30 min on a T4):** runs ParPTE's own feature extractor (Whisper word timings, pauses, hesitations, phoneme accuracy) on every clip, then fits gradient-boosted regressors → `speaking_traits.joblib`.
* **Part B (optional, ≈1.5 h):** fine-tunes `wav2vec2-base` end to end on the raw audio → `speaking_ssl/model.onnx`.

**Notebook settings:** Accelerator **GPU T4 x1** (or P100), Internet **On**. Then *Run all*.
When it finishes, download `parpte_speaking_models.zip` from the Output panel and unzip it into `backend/models/`.

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/chiran-commits/ParPTE.git"
# If the repo is private, add a Kaggle secret named GITHUB_TOKEN (Add-ons → Secrets) with a read-only token.
try:
    from kaggle_secrets import UserSecretsClient
    REPO = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("No GITHUB_TOKEN secret found — cloning as a public repo")

if not os.path.exists("/kaggle/working/ParPTE"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO, "/kaggle/working/ParPTE"], check=True)
sys.path.insert(0, "/kaggle/working/ParPTE/backend")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper>=1.1", "g2p_en", "scikit-learn==1.5.2",
                "datasets>=2.19", "soundfile", "librosa", "onnx", "onnxruntime"], check=True)
import nltk
for pkg in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "cmudict"]:
    nltk.download(pkg, quiet=True)

# Use the GPU for Whisper and the phoneme model while extracting features
os.environ.update(WHISPER_MODEL="small.en", WHISPER_DEVICE="cuda", WHISPER_COMPUTE="float16", PHONEME_DEVICE="cuda")

## 1 · Load speechocean762

In [ ]:
import numpy as np, pandas as pd
from datasets import load_dataset, Audio

ds = load_dataset("mispeech/speechocean762")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
print(ds)
ex = ds["train"][0]
print(ex["text"], "| accuracy", ex["accuracy"], "fluency", ex["fluency"], "prosodic", ex["prosodic"])

## 2 · Part A — extract ParPTE features
`extract_features()` is imported from the app itself, so the model is trained on exactly the numbers the server computes when it scores a student.

In [ ]:
from tqdm.auto import tqdm
from app.scoring.speaking import extract_features, SPEAKING_FEATURES
from app.scoring import asr, phoneme
assert asr.whisper_available(), "Whisper failed to load — is the GPU enabled?"
assert phoneme.available(), "Phoneme model failed to load"

CACHE = "/kaggle/working/speechocean_features.parquet"
if os.path.exists(CACHE):
    df = pd.read_parquet(CACHE)
else:
    rows = []
    for split in ["train", "test"]:
        for ex in tqdm(ds[split], desc=split):
            wave = np.asarray(ex["audio"]["array"], dtype=np.float32)
            f = extract_features(wave, ex["text"])
            rows.append({**{k: float(f[k]) for k in SPEAKING_FEATURES}, "split": split,
                         "accuracy": ex["accuracy"], "fluency": ex["fluency"], "prosodic": ex["prosodic"], "total": ex["total"]})
    df = pd.DataFrame(rows)
    df.to_parquet(CACHE)
df.describe().T

## 3 · Part A — train and evaluate the trait regressors

In [ ]:
from scipy.stats import pearsonr
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib, sklearn

TARGETS = ["accuracy", "fluency", "prosodic"]
tr, te = df[df.split == "train"], df[df.split == "test"]
X_tr, X_te = tr[SPEAKING_FEATURES].values, te[SPEAKING_FEATURES].values

metrics, models = {}, {}
for t in TARGETS:
    m = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=0)
    m.fit(X_tr, tr[t].values)
    pred = m.predict(X_te)
    metrics[t] = {"pcc": float(pearsonr(pred, te[t].values)[0]), "mae": float(np.abs(pred - te[t].values).mean())}
    print(f"{t:9s}  PCC={metrics[t]['pcc']:.3f}  MAE={metrics[t]['mae']:.2f}")
    # refit on all data for the shipped model
    models[t] = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=0).fit(
        df[SPEAKING_FEATURES].values, df[t].values)

OUT = "/kaggle/working/parpte_models"
os.makedirs(OUT, exist_ok=True)
joblib.dump({"version": "1.0", "feature_names": SPEAKING_FEATURES, "models": models, "metrics": metrics,
             "sklearn": sklearn.__version__, "data": "speechocean762"}, f"{OUT}/speaking_traits.joblib")
print("saved", f"{OUT}/speaking_traits.joblib")

For reference, published systems reach roughly PCC 0.70–0.78 for utterance-level accuracy and fluency on speechocean762. Part A trades a little accuracy for being small and CPU-friendly. Part B usually adds a few points.

## 4 · Part B (optional) — fine-tune wav2vec2 on raw audio
Set `RUN_PART_B = True` to train. It needs about 1.5 hours on a T4.

In [ ]:
RUN_PART_B = False

if RUN_PART_B:
    import torch, torch.nn as nn, random
    from transformers import Wav2Vec2Model

    device = "cuda"
    MAX_S = 15

    class Scorer(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
            self.enc.feature_extractor._freeze_parameters()
            self.head = nn.Sequential(nn.Linear(768, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 3))

        def forward(self, x):
            return self.head(self.enc(x).last_hidden_state.mean(1))

    def prep(ex):
        w = np.asarray(ex["audio"]["array"], dtype=np.float32)[: 16000 * MAX_S]
        return (w - w.mean()) / (w.std() + 1e-7), np.array([ex[t] for t in TARGETS], dtype=np.float32)

    train = [prep(e) for e in tqdm(ds["train"], desc="prep train")]
    test = [prep(e) for e in tqdm(ds["test"], desc="prep test")]

    def batches(data, bs, shuffle):
        idx = list(range(len(data)))
        if shuffle: random.shuffle(idx)
        for i in range(0, len(idx), bs):
            items = [data[j] for j in idx[i:i + bs]]
            L = max(len(w) for w, _ in items)
            x = np.stack([np.pad(w, (0, L - len(w))) for w, _ in items])
            y = np.stack([t for _, t in items])
            yield torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

    model = Scorer().to(device)
    opt = torch.optim.AdamW([{"params": model.enc.parameters(), "lr": 3e-5}, {"params": model.head.parameters(), "lr": 1e-3}], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    EPOCHS = 6
    for ep in range(EPOCHS):
        model.train()
        for x, y in tqdm(batches(train, 8, True), total=len(train) // 8, desc=f"epoch {ep+1}"):
            with torch.autocast("cuda", dtype=torch.float16):
                loss = nn.functional.mse_loss(model(x).float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        model.eval(); preds, ys = [], []
        with torch.no_grad():
            for x, y in batches(test, 1, False):
                preds.append(model(x).float().cpu().numpy()[0]); ys.append(y.cpu().numpy()[0])
        preds, ys = np.array(preds), np.array(ys)
        print(f"epoch {ep+1}: " + "  ".join(f"{t} PCC={pearsonr(preds[:, i], ys[:, i])[0]:.3f}" for i, t in enumerate(TARGETS)))

    # ---- export to ONNX for CPU inference on the server
    import json, onnxruntime as ort
    os.makedirs(f"{OUT}/speaking_ssl", exist_ok=True)
    model = model.float().cpu().eval()
    dummy = torch.randn(1, 16000 * 4)
    torch.onnx.export(model, dummy, f"{OUT}/speaking_ssl/model.onnx", input_names=["input_values"], output_names=["scores"],
                      dynamic_axes={"input_values": {0: "batch", 1: "samples"}, "scores": {0: "batch"}}, opset_version=17, dynamo=False)
    json.dump({"targets": TARGETS, "max_seconds": 30, "base_model": "facebook/wav2vec2-base",
               "pcc": {t: float(pearsonr(preds[:, i], ys[:, i])[0]) for i, t in enumerate(TARGETS)}},
              open(f"{OUT}/speaking_ssl/config.json", "w"), indent=2)
    sess = ort.InferenceSession(f"{OUT}/speaking_ssl/model.onnx")
    w, _ = test[0]
    print("onnx:", sess.run(None, {"input_values": w[None, :]})[0], " torch:", model(torch.from_numpy(w)[None]).detach().numpy())

## 5 · Package for download

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/parpte_speaking_models", "zip", OUT)
print("Download parpte_speaking_models.zip from the Output panel and unzip it into backend/models/")